# Capítulo 1 — Leer los datos · Notebook para completar

*Matemáticas para la IA, paso a paso*, Belk Siri — notebook complementario.

[Abrir en Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/es/ch01_a_completer.ipynb) · Página complementaria: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/es.html

Las celdas que debes modificar llevan la indicación **PARA COMPLETAR**. Ejecuta las celdas en orden, de arriba abajo: mientras falte una respuesta, las celdas de verificación lo indican sin producir un error. La solución está en `ch01_corrige.ipynb`, en la misma página complementaria.

**Convención.** Como en el libro, la varianza, la desviación típica, la puntuación z y la covarianza dividen por $n$ (`ddof=0`), salvo que se indique lo contrario.

**Índice.** Datos · Los cálculos de las lecciones · Ejercicios 1.1 a 1.6

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 2.6)
# Colores del libro: un color = un papel (observado, predicho, desviación, recta del colega)
BLUE, ORANGE, RESIDUAL, GREEN = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Datos

Los identificadores (D0.1, D1.1…) son los del libro y de los archivos CSV.

In [ ]:
# Datos del capítulo 1, copiados de los archivos CSV del libro (no modificar)

# D0.1 — las seis viviendas de la apertura
homes = pd.DataFrame({
    'home':   ['A', 'B', 'C', 'D', 'E', 'F'],
    'floor_area': [32, 45, 54, 70, 85, 98],
    'price':  [118, 172, 185, 245, 279, 335],
})

# D1.1 — precios de venta de los T2 (dos habitaciones, k€), lección 1.1
north = [174, 177, 181, 183, 185]
south  = [150, 165, 185, 195, 205]

# D1.2 — registro de ventas del barrio Sur, lecciones 1.2 a 1.5
sales_log = pd.DataFrame({
    'sale':       ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'S13'],
    'rooms':      [1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 4, 4, 4],
    'floor_area':  [24, 27, 43, 40, 44, 46, 52, 61, 64, 70, 82, 88, 95],
    'energy_class':         ['D', 'C', 'F', 'D', 'C', 'D', 'C', 'E', 'D', 'B', 'E', 'C', 'G'],
    'price':   [96, 108, 150, 165, 185, 195, 205, 228, 241, 276, 289, 318, 260],
    'days_to_sell': [9, 18, 97, 21, 26, 23, 31, 14, 35, 28, 42, 58, 183],
})

# D1.5 — precio por m² según la planta (k€/m²), lección 1.5
floors = pd.DataFrame({
    'floor':        [0, 1, 2, 3, 4, 5, 6],
    'price_per_m2': [3.0, 3.3, 3.5, 3.6, 3.5, 3.3, 3.0],
})

# D1.E1 — plazos de venta de tres agencias (días), ejercicio 1.1
agency_delays = {'P': [28, 29, 30, 31, 32], 'Q': [20, 25, 30, 35, 40], 'R': [10, 28, 30, 32, 50]}

# D1.E2 — alquileres mensuales de los estudios (€), ejercicio 1.2
tilleuls = [585, 590, 600, 605, 605, 615]
port     = [520, 560, 590, 620, 640, 670]

### Las variables de los datos

Los nombres del código están en inglés. Para las columnas, la última columna da el nombre utilizado en los archivos CSV del libro.

| Nombre | Significado | Unidad | Columna del CSV |
|:--|:--|:--|:--|
| `home` | identificador de la vivienda (A, B…) | — | `logement` |
| `floor_area` | superficie habitable de la vivienda | m² | `surface_m2` |
| `price` | precio de venta | k€ | `prix_kEUR` |
| `sale` | identificador de la venta (S01…) | — | `vente` |
| `rooms` | número de habitaciones | habitaciones | `pieces` |
| `energy_class` | clase del certificado de eficiencia energética (A a G) | — | `dpe` |
| `days_to_sell` | plazo de venta | días | `delai_jours` |
| `floor` | planta | — | `etage` |
| `price_per_m2` | precio por metro cuadrado | k€/m² | `prix_m2_kEUR` |

In [ ]:
# Herramientas proporcionadas: no tienes que modificarlas

def book_quartiles(values):
    """Q1 y Q3 según la regla del libro (ecuación 1.5): rangos ⌈n/4⌉ y ⌈3n/4⌉."""
    x = sorted(values)
    n = len(x)
    rank_q1 = -(-n // 4)          # n/4 redondeado al entero superior
    rank_q3 = -(-3 * n // 4)      # 3n/4 redondeado al entero superior
    return x[rank_q1 - 1], x[rank_q3 - 1]   # rango 1 = posición 0 en Python


def dot_rows(series, xlabel, bands=None, ax=None):
    """Una fila de puntos por serie, como en las figuras de la lección 1.1.
    bands: diccionario {nombre: (media, desviación típica)} para dibujar media ± desviación típica."""
    if ax is None:
        fig, ax = plt.subplots()
    names = list(series)
    colors = [BLUE, ORANGE, '#52514e']
    for i, name in enumerate(names):
        y = len(names) - 1 - i
        if bands and name in bands:
            m, s = bands[name]
            ax.add_patch(plt.Rectangle((m - s, y - 0.25), 2 * s, 0.5, color='#e9e8e4'))
            ax.plot([m, m], [y - 0.3, y + 0.3], color='black')
        ax.scatter(series[name], [y] * len(series[name]), s=50,
                   color=colors[i % 3], zorder=3)
    ax.set_yticks(range(len(names)), list(reversed(names)))
    ax.set_ylim(-0.6, len(names) - 0.4)
    ax.set_xlabel(xlabel)
    ax.grid(axis='x', alpha=0.4)
    return ax

## Los cálculos de las lecciones, en Python

Estas celdas rehacen los cálculos de las lecciones 1.1 a 1.5. Ejecútalas y compáralas con el libro. Aquí, `np.std` y `np.var` se llaman con `ddof=0`, es decir, con una división por $n$, como en el capítulo.

In [ ]:
# Lección 1.1 — media, varianza y desviación típica (división por n)
for name, one_series in [('Norte', north), ('Sur', south)]:
    print(f'{name}: media = {np.mean(one_series):g} k€, varianza = {np.var(one_series, ddof=0):g} k€², '
          f'desviación típica = {np.std(one_series, ddof=0):g} k€')

In [ ]:
# Lección 1.2 — frecuencias por clase energética (variable cualitativa ordinal)
sales_log['energy_class'].value_counts().reindex(list('ABCDEFG'), fill_value=0)

In [ ]:
# Lección 1.3 — media, mediana y cuartiles de los plazos
d = sales_log['days_to_sell']
print('media:', d.mean(), ' mediana:', d.median())
print('sin S13: media', d[sales_log['sale'] != 'S13'].mean(),
      ' mediana', d[sales_log['sale'] != 'S13'].median())
print('cuartiles, regla del libro (ecuación 1.5):', book_quartiles(d))
print('cuartiles, NumPy por defecto (interpolación):', np.quantile(d, [0.25, 0.75]))

In [ ]:
# Lección 1.4 — puntuaciones estandarizadas (ecuación 1.6, desviación típica con división por n)
for name, one_series, sale in [('Norte', north, 188), ('Sur', south, 210)]:
    m, s = np.mean(one_series), np.std(one_series, ddof=0)
    print(f'{name}: z de la serie = {np.round((np.array(one_series) - m) / s, 4)}, '
          f'z de la venta a {sale} k€ = {(sale - m) / s:g}')

In [ ]:
# Lección 1.5 — covarianza y correlación de los cinco T2 del Sur (división por n)
t2 = sales_log[sales_log['rooms'] == 2]
x, y = t2['floor_area'].to_numpy(float), t2['price'].to_numpy(float)
cov = np.mean((x - x.mean()) * (y - y.mean()))
print('covarianza:', cov, 'm²·k€   r:', cov / (x.std(ddof=0) * y.std(ddof=0)))
print('r con np.corrcoef:', np.corrcoef(x, y)[0, 1].round(4))
print('r planta / precio por m²:', np.corrcoef(floors['floor'], floors['price_per_m2'])[0, 1].round(4))

## Ejercicio 1.1 — Leer un gráfico

*Datos D1.E1. Conceptos: media y desviación típica (lección 1.1).* El gráfico de abajo reproduce el del ejercicio 1.1 del libro.

In [ ]:
dot_rows({f'Agencia {a}': v for a, v in agency_delays.items()},
                  'Plazo de venta (días)')
plt.xlim(0, 60)
plt.show()

**a) y b)** Estima a ojo el plazo medio de cada agencia y luego ordena las agencias de la menos dispersa a la más dispersa.

In [ ]:
# PARA COMPLETAR: tus estimaciones a ojo, antes de cualquier cálculo
estimated_mean = {'P': None, 'Q': None, 'R': None}   # en días
spread_order = None   # lista, de la menos a la más dispersa, p. ej. ['Q', 'P', 'R']

In [ ]:
# Verificación (división por n)
for a, v in agency_delays.items():
    print(f'Agencia {a}: media = {np.mean(v):g} d, desviación típica = {np.std(v, ddof=0):.2f} d')
true_order = sorted(agency_delays, key=lambda a: np.std(agency_delays[a]))
if spread_order is None or None in estimated_mean.values():
    print('Completa la celda anterior.')
else:
    print('Orden de dispersión:', 'correcto' if spread_order == true_order else 'revisar')

**c)** Cuenta, para Q y para R, las ventas cerradas a 2 días o menos de la media.

In [ ]:
# PARA COMPLETAR: tus recuentos, leídos en el gráfico
close_to_Q = None
close_to_R = None

In [ ]:
# Verificación
for a, yours in [('Q', close_to_Q), ('R', close_to_R)]:
    v = np.array(agency_delays[a])
    truth = int(np.sum(np.abs(v - v.mean()) <= 2))
    if yours is None:
        print(f'Agencia {a}: aún sin completar.')
    else:
        print(f"Agencia {a}: {'correcto' if yours == truth else 'revisar'}")

*Tu explicación para c):* …

## Ejercicio 1.2 — Dos residencias, un mismo alquiler medio

*Datos D1.E2. Conceptos: media, varianza y desviación típica (lección 1.1).* Camille no puede pagar más de 570 € al mes. Las preguntas a) a d) se hacen primero en papel, con la tabla del ejercicio 1.2 del libro. Este notebook sirve para verificar tus respuestas, después para programar la desviación típica (pregunta e) y comparar dos convenciones (pregunta f).

In [ ]:
dot_rows({'Les Tilleuls': tilleuls, 'Le Port': port}, "Alquiler mensual de un estudio (€)")
plt.axvline(570, color='black', linestyle=':', label='presupuesto de Camille: 570 €')
plt.xlim(500, 700)
plt.legend(loc='upper right', fontsize=8)
plt.show()

### a) Leer los alquileres

¿Cuántos estudios puede alquilar Camille en cada residencia con 570 €, y luego con 620 €?

In [ ]:
# PARA COMPLETAR: número de estudios asequibles, leído en la lista de alquileres
affordable = {
    570: {'Les Tilleuls': None, 'Le Port': None},
    620: {'Les Tilleuls': None, 'Le Port': None},
}

In [ ]:
# Verificación: `<=` produce verdadero/falso, np.sum cuenta los «verdadero»
for budget, answers in affordable.items():
    for name, one_series in [('Les Tilleuls', tilleuls), ('Le Port', port)]:
        truth = int(np.sum(np.array(one_series) <= budget))
        yours = answers[name]
        status = 'aún sin completar' if yours is None else ('correcto' if yours == truth else 'revisar')
        print(f'Presupuesto {budget} €, {name}: {status}')

*Tu respuesta a a), en una frase:* …

### b) Las dos medias

Calcúlalas en papel y después anota tus resultados abajo.

In [ ]:
# PARA COMPLETAR: tus medias calculadas a mano (en €)
mean_tilleuls_by_hand = None
mean_port_by_hand = None

In [ ]:
# Verificación
for name, one_series, yours in [('Les Tilleuls', tilleuls, mean_tilleuls_by_hand),
                              ('Le Port', port, mean_port_by_hand)]:
    if yours is None:
        print(f'{name}: completa la celda anterior.')
    else:
        ok = np.isclose(yours, np.mean(one_series))
        print(f"{name}: {'correcto' if ok else 'revisar'} (tu valor: {yours} €)")

### c) Varianza y desviación típica a mano (división por $n = 6$)

En papel, prepara para cada residencia la tabla de las desviaciones respecto a la media y de sus cuadrados (modelo: la tabla del cálculo paso a paso de la lección 1.1). Anota aquí los cuatro resultados.

In [ ]:
# PARA COMPLETAR: tus resultados calculados a mano (división por n)
variance_tilleuls_by_hand = None   # en €²
variance_port_by_hand = None       # en €²
std_tilleuls_by_hand = None # en €
std_port_by_hand = None     # en €

In [ ]:
# Verificación (división por n: ddof=0)
your_values = {
    'varianza Tilleuls': (variance_tilleuls_by_hand, np.var(tilleuls, ddof=0)),
    'varianza Port': (variance_port_by_hand, np.var(port, ddof=0)),
    'desviación típica Tilleuls': (std_tilleuls_by_hand, np.std(tilleuls, ddof=0)),
    'desviación típica Port': (std_port_by_hand, np.std(port, ddof=0)),
}
for name, (yours, expected) in your_values.items():
    if yours is None:
        print(f'{name}: aún sin completar.')
    else:
        print(f"{name}: {'correcto' if np.isclose(yours, expected) else 'revisar'}")

### d) Interpretar

*Tu respuesta:* …

### e) Programar la desviación típica

Completa la función siguiendo la ecuación 1.3 del libro, con una división por $n$:

$$\sigma = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})^2}$$

No uses ni `np.std` ni `np.var`: el objetivo es que escribas tú mismo cada paso.

In [ ]:
# Ejercicio 1.2 e) — PARA COMPLETAR
def std_dev(values):
    """Desviación típica, división por n (ecuación 1.3)."""
    x = np.array(values, dtype=float)
    n = len(x)
    mean = None   # PARA COMPLETAR: ecuación 1.1
    deviations = None    # PARA COMPLETAR: una desviación por valor
    variance = None  # PARA COMPLETAR: media de los cuadrados
    return None      # PARA COMPLETAR: vuelta a la unidad original

<details><summary><b>Pista</b> (haz clic si te atascas)</summary>

Con NumPy, una operación escrita sobre un array se aplica a cada valor: `x - mean` da de una vez las seis desviaciones, y `deviations ** 2` sus seis cuadrados. `np.sum(...)` suma, `np.sqrt(...)` calcula la raíz cuadrada.

</details>

In [ ]:
# Verificación: ejecuta esta celda sin modificarla
if std_dev(tilleuls) is None:
    print('La función todavía devuelve None: complétala.')
else:
    for name, one_series, expected in [('Les Tilleuls', tilleuls, 10.0), ('Le Port', port, 50.0)]:
        obtained = std_dev(one_series)
        status = 'correcto' if np.isclose(obtained, expected) else f'revisar (esperado: {expected})'
        print(f'{name}: {obtained:.2f} € → {status}')

### f) Para profundizar: dividir por $n$ o por $n - 1$

Ejecuta la celda de abajo. Obtén 10,95 a mano dividiendo la suma de los cuadrados por $n - 1 = 5$. ¿Qué opción de pandas da la convención del capítulo?

*Tu respuesta:* …

In [ ]:
rents = pd.DataFrame({'Les Tilleuls': tilleuls, 'Le Port': port})
print('pandas, por defecto (división por n − 1):', rents.std().round(2).to_dict())
print('pandas, ddof=0 (división por n)         :', rents.std(ddof=0).round(2).to_dict())

## Ejercicio 1.3 — ¿Una venta sorprendente?

*Datos D1.1. Conceptos: puntuación estandarizada (lección 1.4, ecuación 1.6).* Un T2 (piso de dos habitaciones) acaba de venderse por 196 k€. Medias y desviaciones típicas: las de la lección 1.1, con división por $n$.

In [ ]:
# PARA COMPLETAR: tus cálculos hechos en papel
z_north = None
z_south = None
south_price_as_unusual = None   # en k€, pregunta c)

In [ ]:
# Verificación (desviación típica con división por n)
expected_values = {
    'z Norte': (z_north, (196 - np.mean(north)) / np.std(north, ddof=0)),
    'z Sur': (z_south, (196 - np.mean(south)) / np.std(south, ddof=0)),
    'precio Sur equivalente': (south_price_as_unusual,
                            np.mean(south) + (196 - np.mean(north)) / np.std(north, ddof=0) * np.std(south, ddof=0)),
}
for name, (yours, truth) in expected_values.items():
    status = 'aún sin completar' if yours is None else ('correcto' if np.isclose(yours, truth) else 'revisar')
    print(f'{name}: {status}')

*Tu respuesta a b):* …

## Ejercicio 1.4 — ¿Van juntos superficie y precio?

*Datos D0.1 (las seis viviendas de la apertura). Conceptos: covarianza y correlación (lección 1.5, ecuaciones 1.7 y 1.8).*

In [ ]:
plt.figure(figsize=(5, 3.5))
plt.scatter(homes['floor_area'], homes['price'], color=BLUE)
for _, l in homes.iterrows():
    plt.annotate(l['home'], (l['floor_area'], l['price']), xytext=(-10, 4), textcoords='offset points')
plt.xlabel('Superficie (m²)')
plt.ylabel('Precio de venta (k€)')
plt.grid(alpha=0.4)
plt.show()

**a)** Antes de cualquier cálculo: ¿$r$ será positivo o negativo? ¿Cercano a 0, a 0,5 o a 1?

*Tu respuesta:* …

**b)** Completa las líneas de abajo. Usa una división por $n$ en todas partes.

In [ ]:
x = homes['floor_area'].to_numpy(dtype=float)
y = homes['price'].to_numpy(dtype=float)
n = len(x)

products = None     # PARA COMPLETAR: (x_i − x̄)(y_i − ȳ) para cada vivienda
covariance = None   # PARA COMPLETAR: ecuación 1.7, división por n
r = None            # PARA COMPLETAR: ecuación 1.8 (desviaciones típicas con ddof=0)

<details><summary><b>Pista</b></summary>

`x.mean()` da la media, `x - x.mean()` las seis desviaciones. El producto `*` de dos arrays multiplica término a término. `x.std(ddof=0)` da la desviación típica con división por $n$.

</details>

In [ ]:
# Verificación
if r is None or covariance is None:
    print('Completa la celda anterior.')
else:
    print('covarianza:', round(covariance, 2), 'm²·k€ →',
          'correcto' if np.isclose(covariance, 1637.5) else 'revisar (esperado: 1637,5)')
    print('r:', round(r, 4), '→', 'correcto' if np.isclose(r, np.corrcoef(x, y)[0, 1]) else 'revisar')

**c)** Una correlación cercana a 1, pero desviaciones de 5 a 11 k€ en valor absoluto: ¿es contradictorio?

*Tu respuesta:* …

In [ ]:
print('np.cov por defecto (división por n − 1):', np.cov(x, y)[0, 1])
print('np.cov con bias=True (división por n)  :', np.cov(x, y, bias=True)[0, 1])

## Ejercicio 1.5 — Predecir y luego verificar

*Datos D1.1. Conceptos: media y desviación típica (lección 1.1), mediana y rango intercuartílico (lección 1.3).* Un sexto T2 se vende en el Norte por 240 k€.

**a)** Antes de cualquier cálculo, anota tus predicciones: ¿la media, la mediana, la desviación típica y el rango intercuartílico aumentan mucho, un poco o nada?

*Tus predicciones:* …

In [ ]:
north_plus = north + [240]
print('Norte:', sorted(north))
print('Norte con la venta a 240 k€:', sorted(north_plus))

**b)** Completa la función `summary`. Debe devolver los cuatro indicadores, con la desviación típica dividida por $n$ y los cuartiles calculados con la regla del libro (`book_quartiles`, proporcionada más arriba). Las claves del diccionario están en inglés: `mean` = media, `median` = mediana, `std_dev` = desviación típica, `iqr` = rango intercuartílico.

In [ ]:
# PARA COMPLETAR
def summary(one_series):
    q1, q3 = book_quartiles(one_series)
    return {
        'mean': None,              # PARA COMPLETAR
        'median': None,              # PARA COMPLETAR: np.median
        'std_dev': None,           # PARA COMPLETAR: división por n
        'iqr': None,  # PARA COMPLETAR: a partir de q1 y q3
    }

In [ ]:
table = pd.DataFrame({'Norte (5 ventas)': summary(north), 'Norte+240 k€ (6 ventas)': summary(north_plus),
                        'Sur (5 ventas)': summary(south)})
if table.isna().any().any():
    print('Completa la función summary.')
else:
    expected = {'mean': (180, 190), 'median': (181, 182), 'std_dev': (4, 22.66),
               'iqr': (6, 8)}
    for k, (before, after) in expected.items():
        ok = np.isclose(table.loc[k].iloc[0], before) and np.isclose(table.loc[k].iloc[1], after, atol=0.005)
        print(f"{k}: {'correcto' if ok else 'revisar'}")
table.round(2)

In [ ]:
# Las dos series según el modelo de la figura «media ± desviación típica» de la lección 1.1
if table.isna().any().any():
    print('Completa la función summary para mostrar el gráfico.')
else:
    dot_rows({'Norte': north, 'Norte + 240': north_plus},
                      'Precio de venta de un T2 (k€)',
                      bands={'Norte': (np.mean(north), np.std(north, ddof=0)),
                              'Norte + 240': (np.mean(north_plus), np.std(north_plus, ddof=0))})
    plt.xlim(140, 250)
    plt.show()

**c)** ¿Se han vuelto los precios del Norte más imprevisibles que los del Sur?

*Tu respuesta:* …

## Ejercicio 1.6 — Desafío: dos rectas, un mismo error medio

*Datos D0.1. Conceptos: desviación con signo y valor absoluto (apertura, lección 1.1), desviación típica (lección 1.1).* Recta trazada a ojo: precio predicho = 3 × superficie + 30. Recta perezosa: precio predicho = 222 k€ para todas las viviendas.

In [ ]:
price = homes['price'].to_numpy(dtype=float)
floor_area = homes['floor_area'].to_numpy(dtype=float)

eye_line_errors = None          # PARA COMPLETAR: observado − predicho, recta trazada a ojo
lazy_line_errors = None    # PARA COMPLETAR: observado − predicho, recta perezosa

In [ ]:
# Los tres indicadores de la pregunta b) (desviación típica: división por n)
if eye_line_errors is None or lazy_line_errors is None:
    print('Completa la celda anterior.')
else:
    indicators = pd.DataFrame({
        name: {'media de las desviaciones con signo': e.mean(),
              'media de las desviaciones en valor absoluto': np.abs(e).mean(),
              'desviación típica de las desviaciones con signo': e.std(ddof=0)}
        for name, e in [('recta a ojo', eye_line_errors), ('recta perezosa', lazy_line_errors)]
    })
    print('desviaciones, recta a ojo    :', eye_line_errors)
    print('desviaciones, recta perezosa :', lazy_line_errors)
    print('verificación:', 'correcto' if np.allclose(indicators.iloc[:, 0], [1/3, 44/6, 7.5645], atol=1e-3)
          and np.allclose(indicators.iloc[:, 1], [1/3, 64, 72.1472], atol=1e-3) else 'revisar')
    display(indicators.round(2))

In [ ]:
if eye_line_errors is not None and lazy_line_errors is not None:
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.axhline(0, color='black', linewidth=1)
    ax.scatter(floor_area - 0.8, eye_line_errors, color=ORANGE, label="recta a ojo")
    ax.scatter(floor_area + 0.8, lazy_line_errors, color=BLUE, marker='s', label='recta perezosa')
    ax.set_xlabel('Superficie (m²)')
    ax.set_ylabel('Desviación observado − predicho (k€)')
    ax.legend()
    ax.grid(alpha=0.4)
    plt.show()

*Tus respuestas a c) y d):* …

In [ ]:
# Control de d): la desviación típica de las desviaciones de la recta perezosa es igual a la de los precios
if lazy_line_errors is not None:
    print(round(lazy_line_errors.std(ddof=0), 4), round(price.std(ddof=0), 4))